<a href="https://colab.research.google.com/github/estdanielscl/deeplearning/blob/main/proyectofinalcorte2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch torchvision scikit-learn matplotlib seaborn

IMPORTS

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix

DATASET

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.GaussianBlur(3),
    transforms.ToTensor(),
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

train_dataset = ImageFolder("data/train", transform=train_transforms)#datasetss
val_dataset = ImageFolder("data/val", transform=val_transforms)
test_dataset = ImageFolder("data/test", transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

FileNotFoundError: [Errno 2] No such file or directory: 'data/train'

FOCAL LOSS

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2):
        super().__init__()
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss()

    def forward(self, outputs, targets):
        ce_loss = self.ce(outputs, targets)
        pt = torch.exp(-ce_loss)
        loss = (1 - pt) ** self.gamma * ce_loss
        return loss

MODELOS


4.1 Resnet50(base)

In [ ]:
class ResNet50Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = models.resnet50(pretrained=True)

        for param in self.model.parameters():
            param.requires_grad = False

        self.model.fc = nn.Linear(self.model.fc.in_features, 2)

    def forward(self, x):
        return self.model(x)

4.2 EFFICIENT NET

In [ ]:
class EfficientNetModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = models.efficientnet_b0(pretrained=True)

        for param in self.model.parameters():
            param.requires_grad = False

        self.model.classifier[1] = nn.Linear(self.model.classifier[1].in_features, 2)

    def forward(self, x):
        return self.model(x)

FOURIER Y CNN

In [ ]:
def apply_fft(x):
    fft = torch.fft.fft2(x)
    fft = torch.fft.fftshift(fft)
    return torch.abs(fft)

In [ ]:
class FourierCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.fc = nn.Sequential(
            nn.Linear(32 * 56 * 56, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = apply_fft(x)
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

MODELO PROPUESTO CON FFT

In [ ]:
class HybridModel(nn.Module):
    def __init__(self):
        super().__init__()

        # ResNet
        self.resnet = models.resnet50(pretrained=True)
        for param in self.resnet.parameters():
            param.requires_grad = False
        self.resnet.fc = nn.Identity()

        # Fourier branch
        self.freq_conv = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.freq_fc = nn.Linear(32 * 56 * 56, 256)

        # Final classifier
        self.classifier = nn.Sequential(
            nn.Linear(2048 + 256, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 2)
        )

    def forward(self, x):
        spatial = self.resnet(x)

        freq = apply_fft(x)
        freq = self.freq_conv(freq)
        freq = freq.view(freq.size(0), -1)
        freq = self.freq_fc(freq)

        combined = torch.cat((spatial, freq), dim=1)
        return self.classifier(combined)

ENTRENAMIENTO DEL MODELO

In [ ]:
def train_model(model, train_loader, val_loader, epochs=15):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    criterion = FocalLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    train_losses, val_losses = [], []

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        train_losses.append(total_loss)

        # VALIDACIÓN
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

        val_losses.append(val_loss)

        print(f"Epoch {epoch+1}: Train {total_loss:.3f} | Val {val_loss:.3f}")

    return train_losses, val_losses

EVALUACION DEL MODELO

In [ ]:
def evaluate_model(model, loader):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()

    preds, targets = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            preds.extend(predicted.cpu().numpy())
            targets.extend(labels.numpy())

    print(classification_report(targets, preds))

    cm = confusion_matrix(targets, preds)
    sns.heatmap(cm, annot=True, fmt="d")
    plt.title("Confusion Matrix")
    plt.show()

GRAFICAS

In [ ]:
def plot_losses(train_losses, val_losses):
    plt.plot(train_losses, label="Train")
    plt.plot(val_losses, label="Validation")
    plt.legend()
    plt.title("Loss Curve")
    plt.show()

In [ ]:
model = HybridModel()

train_losses, val_losses = train_model(model, train_loader, val_loader)

plot_losses(train_losses, val_losses)

evaluate_model(model, test_loader)

Imagen original ──► ResNet50 ──► Features espaciales
        │
        ▼
FFT (Fourier) ──► CNN ──► Features de frecuencia

           ▼
      Concatenación

           ▼
   Clasificador final